# 3D-RCNN render-and-compare — straight from the **gestalt** repo

Clones [`github.com/davechendatascience/gestalt.git`](https://github.com/davechendatascience/gestalt) and runs the 3D-RCNN miniature using the
repo's own code: PCA shape basis + differentiable renderer + render-and-compare training.
Runs on Colab CPU or GPU.

In [ ]:
import os
if not os.path.isdir('gestalt'):
    !git clone -q https://github.com/davechendatascience/gestalt.git
else:
    !cd gestalt && git pull -q
import sys
sys.path.insert(0, 'gestalt/src')

In [ ]:
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from gestalt.render3d import library
from gestalt.ae3d import rotate3d
from gestalt.r3dcnn import build_shape_basis, render_silhouette, quat_to_R, R3DCNN
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); print('device:', dev)

## Shape basis (PCA modes) over the repo's procedural objects

In [ ]:
D, NC, H = 20, 10, 48
pts = [library()[k][0] for k in library()]
mean, basis, codes = build_shape_basis(pts, D, NC)
mean_t = torch.tensor(mean, device=dev); basis_t = torch.tensor(basis, device=dev)
codes_t = torch.tensor(codes, device=dev)
print('objects:', list(library()), '| shape modes:', NC)

## Synthetic scenes + render-and-compare training (2D mask supervises 3D)

In [ ]:
def rand_quats(n, rng):
    return F.normalize(torch.tensor(rng.normal(size=(n,4)), dtype=torch.float32, device=dev), dim=1)
@torch.no_grad()
def gen(n, seed):
    rng=np.random.default_rng(seed); o=rng.integers(0,len(pts),n)
    gt=codes_t[o]; R=quat_to_R(rand_quats(n,rng))
    occ=(mean_t+gt@basis_t).clamp(0,1).view(n,1,D,D,D); occ_r=rotate3d(occ,R)
    w=torch.linspace(1,0.35,D,device=dev).view(1,1,D,1,1)
    shaded=F.interpolate((occ_r*w).amax(2),size=(H,H),mode='bilinear',align_corners=False)[:,0]
    sil=F.interpolate(1-torch.prod(1-occ_r,2),size=(H,H),mode='bilinear',align_corners=False)[:,0]
    mask=(sil>0.3).float()
    tex=0.6+0.4*torch.tensor(rng.random((n,H,H)),dtype=torch.float32,device=dev)
    img=(shaded*tex+torch.tensor(rng.normal(0,0.04,(n,H,H)),dtype=torch.float32,device=dev)).clamp(0,1)
    return img,mask,gt,R
Xtr,Mtr,Ctr,Rtr=gen(900,0); Xte,Mte,Cte,Rte=gen(120,1)
net=R3DCNN(NC,H).to(dev); opt=torch.optim.Adam(net.parameters(),lr=1.5e-3)
for ep in range(70):
    perm=torch.randperm(len(Xtr))
    for i in range(0,len(Xtr),32):
        idx=perm[i:i+32]; code,R=net(Xtr[idx].unsqueeze(1))
        sil=render_silhouette(code,R,mean_t,basis_t,D,H).clamp(1e-4,1-1e-4)
        loss=F.binary_cross_entropy(sil,Mtr[idx])+0.1*F.mse_loss(code,Ctr[idx])+0.1*F.mse_loss(R,Rtr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    if (ep+1)%20==0: print(f'epoch {ep+1}/70  loss={loss.item():.4f}')

## Results — 2D render-and-compare recovers 3D shape + pose

In [ ]:
net.eval()
with torch.no_grad():
    code,R=net(Xte.unsqueeze(1)); sil=render_silhouette(code,R,mean_t,basis_t,D,H)
    inter=((sil>0.5)&(Mte>0.5)).sum((-1,-2)).float(); union=((sil>0.5)|(Mte>0.5)).sum((-1,-2)).float()
    print('render-and-compare silhouette IoU (test):', float((inter/(union+1e-6)).mean()))
    nv=quat_to_R(rand_quats(6,np.random.default_rng(9))); novel=render_silhouette(code[:6],nv,mean_t,basis_t,D,H)
rows=[(Xte,'input'),(Mte,'GT mask'),(sil,'pred silhouette (R&C)'),(novel,'pred shape, NOVEL pose')]
fig,ax=plt.subplots(4,6,figsize=(9,6))
for r,(A,lab) in enumerate(rows):
    for c in range(6): ax[r,c].imshow(A[c].cpu().numpy(),cmap='gray',vmin=0,vmax=1); ax[r,c].axis('off')
    ax[r,0].set_title(lab,fontsize=8,loc='left')
plt.suptitle('3D-RCNN miniature (from gestalt repo): render-and-compare recovers 3D'); plt.tight_layout(); plt.show()

Row 3 (network's render of its predicted shape+pose) matches row 2 (the GT mask), trained
only by comparing renders to 2D masks; row 4 renders the same predicted shape from a **novel**
pose. Honest limit: 2D silhouettes underdetermine the shape code (single-view is lossy).